# Giai đoạn 2: Phân tích Khám phá Dữ liệu và Trực quan hóa
**Dự án:** Phân tích và Dự đoán Tỷ lệ Tội phạm



In [5]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
load_dotenv()

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [6]:
load_dotenv(dotenv_path="../.env")

data_path = os.getenv("PROCESSED_DATA_PATH")

if data_path:
    print(f"Đang đọc dữ liệu từ đường dẫn cấu hình: {data_path}")
    df = pd.read_csv(data_path)

    print(f"Tải dữ liệu thành công! Tổng số dòng: {len(df)}")
    display(df.head())
else:
    print("Lỗi: Không tìm thấy biến PROCESSED_DATA_PATH trong file .env. Vui lòng kiểm tra lại đường dẫn file .env!")

Đang đọc dữ liệu từ đường dẫn cấu hình: ../data/processed/cleaned_crime_data.csv
Tải dữ liệu thành công! Tổng số dòng: 2215


,communityname,fold,State,pop,perHoush,pctBlack,pctWhite,pctAsian,pctHisp,pct12-21,...,burglaries,burglPerPop,larcenies,larcPerPop,autoTheft,autoTheftPerPop,arsons,arsonsPerPop,violentPerPop,nonViolPerPop
0,BerkeleyHeightstownship,1,NJ,11980,3.10,1.37,91.78,6.50,1.88,12.47,...,14.0,114.85,138.0,1132.08,16.0,131.26,2.0,16.41,41.02,1394.59
1,Marpletownship,1,PA,23123,2.82,0.80,95.57,3.44,0.85,11.01,...,57.0,242.37,376.0,1598.78,26.0,110.55,1.0,4.25,127.56,1955.95
2,Tigardcity,1,OR,29344,2.43,0.74,94.33,3.43,2.35,11.36,...,274.0,758.14,1797.0,4972.19,136.0,376.30,22.0,60.87,218.59,6167.51
3,Gloversvillecity,1,NY,16656,2.40,1.70,97.35,0.50,0.70,12.55,...,225.0,1301.78,716.0,4142.56,47.0,271.93,5.0,21.08,306.64,4425.45
4,Bemidjicity,1,MN,11245,2.76,0.53,89.16,1.17,0.52,24.46,...,91.0,728.93,1060.0,8490.87,91.0,728.93,5.0,40.05,374.06,9988.79


In [7]:
target = 'violentPerPop'

In [ ]:
!pip install plotly

## 1. Bản đồ nhiệt địa lý

### a. Tỷ lệ tội phạm bạo lực tại các bang nước Mỹ

In [ ]:
# 1. Chọn các cột cần thiết và loại bỏ các dòng có giá trị thiếu
crime_EDA_1 = df[["State", "pop", "violentPerPop"]].dropna()

# 2. Tính tổng số vụ tội phạm bạo lực cho từng khu vực
crime_EDA_1["total_violent_crimes"] = (
    crime_EDA_1["violentPerPop"] / 100000
) * crime_EDA_1["pop"]

# 3. Gom nhóm theo bang và tính tổng
state_groups = (
    crime_EDA_1.groupby("State")
    .agg(tot_pop=("pop", "sum"), tot_vio=("total_violent_crimes", "sum"))
    .reset_index()
)

# Tính toán tỷ lệ tội phạm của từng bang
state_groups["crime_rate"] = state_groups["tot_vio"] / state_groups["tot_pop"]

# 4. Tạo thông tin hiển thị
state_groups["hover_text"] = (
    state_groups["State"].astype(str)
    + "<br>pop: "
    + state_groups["tot_pop"].map("{:,}".format)
)

# 5. Vẽ bản đồ tương tác với Plotly Express
fig = px.choropleth(
    state_groups,
    locations="State",
    locationmode="USA-states",
    color="crime_rate",
    hover_name="State",
    color_continuous_scale="Reds",
    labels={"crime_rate": "Tỷ lệ tội phạm"},
    scope="usa",
    title="Tỷ lệ tội phạm bạo lực tại các bang nước Mỹ",
)


fig.update_traces(marker_line_color="white", marker_line_width=2)

# Hiển thị bản đồ p
fig.show()

### b. Tỷ lệ tội phạm không bạo lực tại các bang nước Mỹ

In [ ]:
# 1. Chọn các cột cần thiết và loại bỏ các dòng có giá trị thiếu
crime_EDA_1 = df[["State", "pop", "nonViolPerPop"]].dropna()

# 2. Tính tổng số vụ tội phạm bạo lực cho từng khu vực (mutate)
crime_EDA_1["total_non_violent_crimes"] = (
                                              crime_EDA_1["nonViolPerPop"] / 100000
                                      ) * crime_EDA_1["pop"]

# 3. Gom nhóm theo bang và tính tổng
state_groups = (
    crime_EDA_1.groupby("State")
    .agg(tot_pop=("pop", "sum"), tot_non_vio=("total_non_violent_crimes", "sum"))
    .reset_index()
)

# Tính toán tỷ lệ tội phạm của từng bang
state_groups["non_crime_rate"] = state_groups["tot_non_vio"] / state_groups["tot_pop"]

# 4. Tạo thông tin hiển thị
state_groups["hover_text"] = (
        state_groups["State"].astype(str)
        + "<br>pop: "
        + state_groups["tot_pop"].map("{:,}".format)
)

# 5. Vẽ bản đồ
fig = px.choropleth(
    state_groups,
    locations="State",
    locationmode="USA-states",
    color="non_crime_rate",
    hover_name="State",
    color_continuous_scale="Oranges",
    labels={"non_crime_rate": "Tỷ lệ tội phạm không bạo lực"},
    scope="usa",
    title="Tỷ lệ tội phạm không bạo lực tại các bang nước Mỹ",
)

fig.update_traces(marker_line_color="white", marker_line_width=2)

# Hiển thị bản đồ
fig.show()

### c. Tỷ lệ phần trăm dân số da đen tại các bang nước Mỹ

In [ ]:
import pandas as pd
import plotly.express as px

# 1. Chọn các cột cần thiết và loại bỏ các dòng có giá trị thiếu
crime_EDA_3 = df[["State", "pop", "pctBlack"]].dropna()

# 2. Tính số lượng người da đen cho từng khu vực
# (Do pctBlack được lưu dưới dạng tỷ lệ từ 0 đến 1)
crime_EDA_3["black_pop_absolute"] = crime_EDA_3["pctBlack"] * crime_EDA_3["pop"]

# 3. Gom nhóm theo bang và tính tổng
state_groups = (
    crime_EDA_3.groupby("State")
    .agg(
        tot_pop=("pop", "sum"), tot_black_pop=("black_pop_absolute", "sum")
    )  # Tính tổng số dân và tổng số người da đen
    .reset_index()
)

# Tính tỷ lệ phần trăm dân số da đen của từng bang (%)
state_groups["black_pop_rate"] = (
    state_groups["tot_black_pop"] / state_groups["tot_pop"]
) * 100

# 4. Tạo thông tin hiển thị
state_groups["hover_text"] = (
    state_groups["State"].astype(str)
    + "<br>pop: "
    + state_groups["tot_pop"].map("{:,}".format)
)

# 5. Vẽ bản đồ
fig = px.choropleth(
    state_groups,
    locations="State",
    locationmode="USA-states",
    color="black_pop_rate",
    hover_name="State",
    hover_data={
        "State": False,
        "black_pop_rate": ":.2f",
    },
    color_continuous_scale="Blues",
    labels={"black_pop_rate": "Tỷ lệ dân số da đen (%)"},
    scope="usa",
    title="Tỷ lệ phần trăm dân số da đen tại các bang nước Mỹ",
)


fig.update_traces(marker_line_color="white", marker_line_width=2)

# Hiển thị bản đồ
fig.show()

##2. Bản đồ chữ:

In [ ]:
!pip install wordcloud matplotlib

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from wordcloud import WordCloud

# 1. Chọn các cột cần thiết và loại bỏ các dòng có giá trị thiếu
crime_EDA_3 = df[["communityname", "State", "pop", "violentPerPop"]].dropna()

# 2. Tạo cột 'Place' kết hợp tên cộng đồng và Bang
crime_EDA_3["Place"] = (
    crime_EDA_3["communityname"].astype(str) + ", " + crime_EDA_3["State"].astype(str)
)

# 3. Tính tổng số vụ tội phạm bạo lực tuyệt đối dựa trên cột 'pop' và 'violentPerPop'
crime_EDA_3["total_violent_crimes"] = (
    crime_EDA_3["violentPerPop"] / 100000
) * crime_EDA_3["pop"]

# 4. Gom nhóm theo 'Place' và tính tổng
place_groups = (
    crime_EDA_3.groupby("Place")
    .agg(tot_pop=("pop", "sum"), tot_vio=("total_violent_crimes", "sum"))
    .reset_index()
)

# Tính toán tỷ lệ tội phạm trên 100,000 dân
place_groups["crime_rate"] = (
    place_groups["tot_vio"] / place_groups["tot_pop"]
) * 100000

# 5. Sắp xếp giảm dần và lấy Top 15 cộng đồng nguy hiểm nhất
top_15_communities = place_groups.sort_values(
    by="crime_rate", ascending=False
).head(15)

# 6. Chuyển đổi dữ liệu thành dạng Dictionary để nạp vào Word Cloud
data_dict = dict(zip(top_15_communities["Place"], top_15_communities["crime_rate"]))

# 7. Tạo Word Cloud
greens = plt.cm.get_cmap('Greens')(np.linspace(0.4, 0.9, 128)) # Bỏ các màu quá nhạt
yellows = plt.cm.get_cmap('YlOrBr')(np.linspace(0.5, 0.9, 128))
combined_colors = np.vstack((greens, yellows))
custom_colormap = mcolors.LinearSegmentedColormap.from_list('GreensYellows', combined_colors)

wordcloud = WordCloud(
    width=2000,           # Tăng độ phân giải cho chữ sắc nét
    height=1000,
    background_color="#1a1d20", # Màu nền xám đen đậm
    colormap=custom_colormap, # Sử dụng dải màu tùy chỉnh
    prefer_horizontal=1.0,    # Ưu tiên chữ nằm ngang
    random_state=42,          # Cố định vị trí để so sánh (có thể thay đổi số)
    collocations=False,       # Ngăn hiển thị các từ đi kèm (ví dụ "city city")
    min_font_size=10,         # Cỡ chữ tối thiểu
    max_font_size=160,        # Tăng cỡ chữ tối đa để tạo sự tương phản lớn
    font_path='Arial',        # Sử dụng phông chữ Arial (hoặc một phông đậm, rõ ràng)
    margin=10                  # Lề giữa các từ
).generate_from_frequencies(data_dict)


# 8. Hiển thị
plt.figure(figsize=(15, 8))
plt.imshow(wordcloud, interpolation="bilinear") # Sử dụng bilinear cho chữ mượt mà
plt.axis("off") # Tắt trục
plt.tight_layout(pad=0)
plt.show()

### Câu hỏi 1: Nhóm tuổi nào dễ bị tổn thương bởi tội phạm hơn?
Chúng ta sẽ phân tích các cột cấu trúc tuổi: `pct12-21`, `pct12-29`, `pct16-24`, `pct65up`.

In [ ]:
age_cols = ['pct12-21', 'pct12-29', 'pct16-24', 'pct65up']
age_corr = df[age_cols + [target]].corr()[target].sort_values(ascending=False)
print('Hệ số tương quan giữa các nhóm tuổi và tỷ lệ tội phạm:')
print(age_corr)

# Trực quan hóa mối quan hệ của nhóm có tương quan mạnh nhất
plt.figure(figsize=(8, 5))
sns.regplot(data=df, x='pct12-29', y=target, scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.title('Tác động của tỷ lệ dân số từ 12-29 tuổi lên Tỷ lệ tội phạm bạo lực')
plt.xlabel('Tỷ lệ phần trăm dân số tuổi 12-29')
plt.ylabel('Tội phạm bạo lực trên 100k dân')
plt.show()

### Câu hỏi 2: Sắc tộc có ảnh hưởng đến tỷ lệ tội phạm không?
Sử dụng các đặc trưng về tỷ lệ sắc tộc cấu thành cộng đồng: `pctWhite`, `pctBlack`, `pctAsian`, `pctHisp`.

In [ ]:
race_cols = ['pctWhite', 'pctBlack', 'pctAsian', 'pctHisp']
race_corr = df[race_cols + [target]].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(race_corr, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Ma trận tương quan giữa Thành phần Sắc tộc và Tội phạm')
plt.show()

### Câu hỏi 3: Giáo dục có vai trò làm giảm tỷ lệ tội phạm không?
Sử dụng các chỉ số: `pctLowEdu`, `pctNotHSgrad`, `pctCollGrad`.

In [ ]:
edu_cols = ['pctLowEdu', 'pctNotHSgrad', 'pctCollGrad']
edu_corr = df[edu_cols + [target]].corr()[target]
print('Tương quan Giáo dục:')
print(edu_corr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.regplot(ax=axes[0], data=df, x='pctNotHSgrad', y=target, scatter_kws={'alpha':0.2}, line_kws={'color':'darkred'})
axes[0].set_title('Chưa tốt nghiệp Cấp 3 vs Tội phạm')

sns.regplot(ax=axes[1], data=df, x='pctCollGrad', y=target, scatter_kws={'alpha':0.2}, line_kws={'color':'green'})
axes[1].set_title('Tốt nghiệp Đại học vs Tội phạm')
plt.show()

### Câu hỏi 4: Bất bình đẳng kinh tế đóng góp thế nào vào tỷ lệ tội phạm?
Xem xét mối quan hệ từ góc độ tài chính: Thu nhập trung bình (`medIncome`), tỷ lệ nghèo đói (`pctPoverty`).

In [ ]:
econ_cols = ['medIncome', 'pctPoverty', 'perCapInc']
print(df[econ_cols + [target]].corr()[target])

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='pctPoverty', y=target, alpha=0.4, color='purple')
plt.title('Mối liên hệ giữa Tỷ lệ nghèo đói (Poverty) và Tội phạm')
plt.xlabel('Tỷ lệ dân số sống dưới chuẩn nghèo (%)')
plt.ylabel('Tỷ lệ tội phạm')
plt.show()

### Câu hỏi 5: Sự thay đổi trong dân số nhập cư có ảnh hưởng đáng kể không?
Phân tích tỷ lệ người nhập cư: `pctForeignBorn`, `pctImmig-3`, `pctImmig-10`.

In [ ]:
immig_cols = ['pctForeignBorn', 'pctImmig-3', 'pctImmig-5', 'pctImmig-8', 'pctImmig-10']
print('Tương quan cấu trúc dân số nhập cư với tội phạm:')
print(df[immig_cols + [target]].corr()[target])